# 04. Citizen Profile Extraction Pipeline

Walkthrough of the transformation pipeline:
```text
Natural Language Input -> LLM Extraction -> Structured JSON -> Pydantic Validation -> CitizenProfile
```

### 1. Test Inputs & Scenario Profiles

In [ ]:
test_scenarios = [
    {
        'label': 'Scenario 1: Punjab Farmer',
        'text': 'I am a 42-year-old farmer from Punjab earning 2 lakh annually.',
        'mock_output': {'age': 42, 'state': 'Punjab', 'occupation': 'farmer', 'annual_income': 200000, 'farmer_status': True}
    },
    {
        'label': 'Scenario 2: Student',
        'text': 'I am a student looking for financial assistance for education.',
        'mock_output': {'student_status': True, 'occupation': 'student'}
    },
    {
        'label': 'Scenario 3: Senior Citizen',
        'text': 'I am 68 years old and retired in a rural village.',
        'mock_output': {'age': 68, 'rural_or_urban': 'rural', 'employment_status': 'unemployed'}
    }
]

### 2. Validation & Normalization with Pydantic

In [ ]:
from app.models.profile import CitizenProfile

for sc in test_scenarios:
    print(f'=== {sc["label"]} ===')
    print(f'Raw Input: "{sc["text"]}"')
    profile = CitizenProfile(**sc['mock_output'])
    print(f'Validated Profile: {profile.get_known_fields()}')
    print(f'Missing Critical: {profile.get_missing_critical_fields()}\n')

### 3. Incremental Profile Merging
When a citizen answers follow-up questions, new information merges without overwriting verified facts.

In [ ]:
existing_profile = CitizenProfile(age=42, state='Punjab', occupation='farmer')
new_disclosure = {'annual_income': 200000, 'land_ownership': 3.5}

merged_dict = existing_profile.model_dump()
merged_dict.update({k: v for k, v in new_disclosure.items() if v is not None})
updated_profile = CitizenProfile(**merged_dict)
print('Updated Profile after follow-up:')
print(updated_profile.get_known_fields())

### Interpretation
Natural language is safely turned into structured objects. Missing facts remain explicitly `None` rather than fabricated.